# Treinamento de modelos

In [ ]:
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torchvision
from torchvision import transforms

import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
import matplotlib.pyplot as plt

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Semente: {seed}")
print(f"Dispositivo: {device}")
print("Versão do Torch:", torch.__version__)
print("Versão do Torchvision:", torchvision.__version__)

student_run_tag = "BZA_11_06_2026"
output_dir = Path("finalProject_outputs")
output_dir.mkdir(exist_ok=True)
(output_dir / student_run_tag).mkdir(exist_ok=True)
print("Diretório de saída:", output_dir / student_run_tag)

Semente: 42
Dispositivo: cuda
Versão do Torch: 2.11.0+cu128
Versão do Torchvision: 0.26.0+cu128
Diretório de saída: finalProject_outputs/BZA_11_06_2026


## Importação dos dados

In [ ]:
from isic2018_dataset import download_isic2018, get_dataloaders

IMAGE_SIZE = 224
BATCH_SIZE = 32

download_isic2018(root="./data/isic2018")


train_loader, val_loader, test_loader = get_dataloaders(
    root="./data/isic2018",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    train_transform=transforms.ToTensor(),
    val_transform=transforms.ToTensor(),
    test_transform=transforms.ToTensor()
)

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader, ConcatDataset

dataset = ConcatDataset([train_loader.dataset, val_loader.dataset])

labels = []
for ds in dataset.datasets:
    if hasattr(ds, "targets"):
        labels.extend(list(ds.targets))
    elif hasattr(ds, "labels"):
        labels.extend(list(ds.labels))
    else:
        # fallback (pode ser caro pois itera o dataset)
        labels.extend([lbl for _, lbl in ds])

# Stratified split no dataset concatenado
train_idx, val_idx = train_test_split(
    list(range(len(dataset))),
    test_size=0.15,
    stratify=labels,
    random_state=seed,
)

train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

## Funções de treinamento e avaliação

### Funções de treinamento

In [ ]:
from tqdm import tqdm

def train_one_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(data_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Exibe loss e acc
        preds = outputs.argmax(dim=1)
        acc = (preds == labels).float().mean()
        print(f"Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

        running_loss += loss.item()
    return running_loss / len(data_loader)

In [ ]:
def train(model, loader, criterion, optimizer, device, num_epochs):
    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, loader, optimizer, criterion, device)
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {train_loss:.4f}")

### Funções de avaliação

In [ ]:
def compute_metrics(outputs, labels):
    """
    Computa métricas relevantes para classificação como
    acurácia, precisão, recall e F1-score.
    """
    preds = outputs.argmax(dim=1)
    accuracy = (preds == labels).float().mean()
    precision = torch.sum((preds == 1) & (labels == 1)) / (torch.sum(preds == 1) + 1e-8)
    recall = torch.sum((preds == 1) & (labels == 1)) / (torch.sum(labels == 1) + 1e-8)
    f1_score = 2 * (precision * recall) / (precision + recall + 1e-8)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

In [ ]:
def evaluate(model, loader):
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            compute_metrics(outputs, labels)


In [ ]:
def plot_roc(model, loader, device, type='binary'):
    """
    Plota a curva ROC para o modelo.
    """
    from sklearn.metrics import roc_curve, auc
    from sklearn.preprocessing import label_binarize

    y_true = []
    y_scores = []

    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            y_true.extend(labels.cpu().numpy())
            y_scores.extend(probs.cpu().numpy())

    if type == 'multiclass':
        n_classes = 7
        classes = np.arange(n_classes)

        # 2. Binarize the true labels for OvR evaluation
        y_test_bin = label_binarize(y_true, classes=classes)

        # 3. Compute ROC curve and ROC area for each of the 7 classes
        fpr = dict()
        tpr = dict()
        roc_auc = dict()

        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_scores[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        # 4. Compute micro-average ROC curve (Optional)
        fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_scores.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

        # 5. Plot all 7 ROC curves
        plt.figure(figsize=(10, 8))

        # Plot micro-average
        plt.plot(fpr["micro"], tpr["micro"],
                label=f'Micro-average ROC (AUC = {roc_auc["micro"]:.2f})',
                color='deeppink', linestyle=':', linewidth=4)

        # Plot each individual class curve
        colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black']
        for i, color in zip(range(n_classes), colors):
            plt.plot(fpr[i], tpr[i], color=color, lw=2,
                    label=f'Class {i} ROC (AUC = {roc_auc[i]:.2f})')

        # Plot baseline random guess line
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Multi-class One-vs-Rest (OvR) ROC Curve')
        plt.legend(loc="lower right")
        plt.grid(True)
        plt.show()
    
    else:

        fpr, tpr, thresholds = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Random guess line
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic (ROC)')
        plt.legend(loc="lower right")
        plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# 1. Setup your data (Replace with your actual 7-class data)
# y_test: True labels (integers 0 to 6)
# y_score: Predicted probabilities from your model, shape (n_samples, 7)
n_classes = 7
classes = np.arange(n_classes)

# 2. Binarize the true labels for OvR evaluation
y_test_bin = label_binarize(y_test, classes=classes)

# 3. Compute ROC curve and ROC area for each of the 7 classes
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# 4. Compute micro-average ROC curve (Optional)
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# 5. Plot all 7 ROC curves
plt.figure(figsize=(10, 8))

# Plot micro-average
plt.plot(fpr["micro"], tpr["micro"],
         label=f'Micro-average ROC (AUC = {roc_auc["micro"]:.2f})',
         color='deeppink', linestyle=':', linewidth=4)

# Plot each individual class curve
colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black']
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'Class {i} ROC (AUC = {roc_auc[i]:.2f})')

# Plot baseline random guess line
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class One-vs-Rest (OvR) ROC Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


## Modelo baseline CNN